# Cognifyz Data Science Internship
## Level 1: Data Exploration, Descriptive Analysis & Geospatial Analysis

**Author:** Data Science Intern  
**Domain:** Data Science  
**Dataset:** `Dataset .csv`

---

### Objectives:
1. **Task 1: Data Exploration and Preprocessing**
   - Identify dataset dimensions (rows and columns).
   - Check and handle missing values in all attributes.
   - Perform data type conversions where necessary.
   - Analyze target variable (`Aggregate rating`) distribution and class imbalances.
2. **Task 2: Descriptive Analysis**
   - Calculate basic statistical measures (mean, median, std, min, max, skewness) for numerical features.
   - Explore categorical variables distribution (`Country Code`, `City`, `Cuisines`).
   - Identify top cuisines and top cities with highest restaurant counts.
3. **Task 3: Geospatial Analysis**
   - Visualize restaurant locations using geographic coordinates (`Latitude`, `Longitude`).
   - Analyze restaurant distributions across cities and countries.
   - Determine correlation between geographic location and aggregate ratings.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Visualization configuration
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
os.makedirs('output_plots', exist_ok=True)


## Task 1: Data Exploration and Preprocessing

In [ ]:
# 1.1 Load dataset
df = pd.read_csv('Dataset .csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


In [ ]:
# 1.2 Data types and structure
df.info()


In [ ]:
# 1.3 Missing Values Check & Imputation
missing_summary = df.isnull().sum()
print("Missing values per column:")
print(missing_summary[missing_summary > 0])

# Impute missing values in Cuisines column
if df['Cuisines'].isnull().sum() > 0:
    df['Cuisines'] = df['Cuisines'].fillna('Unknown')
    print("\nMissing values in 'Cuisines' handled successfully!")

print(f"Total Duplicate Rows: {df.duplicated().sum()}")


In [ ]:
# 1.4 Target Variable ('Aggregate rating') Analysis
print("Aggregate rating statistics:")
print(df['Aggregate rating'].describe())

zero_ratings = (df['Aggregate rating'] == 0).sum()
print(f"\nUnrated / Zero Rating Count: {zero_ratings} ({(zero_ratings/len(df))*100:.2f}%)")

# Visualize Rating Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Aggregate rating'], bins=20, kde=True, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Aggregate Rating Distribution (All Data)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Aggregate Rating')
axes[0].set_ylabel('Frequency')

sns.histplot(df[df['Aggregate rating'] > 0]['Aggregate rating'], bins=20, kde=True, ax=axes[1], color='#1b9e77')
axes[1].set_title('Aggregate Rating Distribution (Excluding 0.0 Unrated)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Aggregate Rating')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


## Task 2: Descriptive Analysis

In [ ]:
# 2.1 Numerical summary statistics
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
desc_table = df[num_cols].describe().T
desc_table['median'] = df[num_cols].median()
desc_table['skewness'] = df[num_cols].skew()
desc_table[['count', 'mean', 'std', 'min', 'median', 'max', 'skewness']]


In [ ]:
# 2.2 Categorical Variables Distribution & Top Entities
print(f"Total Countries: {df['Country Code'].nunique()}")
print(f"Total Cities: {df['City'].nunique()}")
print(f"Total Unique Cuisines Entries: {df['Cuisines'].nunique()}")

# Top 10 Cities
top_cities = df['City'].value_counts().head(10)
print("\nTop 10 Cities by Restaurant Count:\n", top_cities)

# Top 10 Cuisines
cuisines_exploded = df['Cuisines'].str.split(', ').explode()
top_cuisines = cuisines_exploded.value_counts().head(10)
print("\nTop 10 Cuisines by Frequency:\n", top_cuisines)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=top_cities.values, y=top_cities.index, ax=axes[0], palette='Blues_r', hue=top_cities.index, legend=False)
axes[0].set_title('Top 10 Cities with Highest Restaurant Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Restaurants')

sns.barplot(x=top_cuisines.values, y=top_cuisines.index, ax=axes[1], palette='Greens_r', hue=top_cuisines.index, legend=False)
axes[1].set_title('Top 10 Cuisines Offered', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Restaurant Count')

plt.tight_layout()
plt.show()


## Task 3: Geospatial Analysis

In [ ]:
# 3.1 Geospatial Scatter Plot
geo_valid = df[(df['Latitude'] != 0) | (df['Longitude'] != 0)]
print(f"Valid Coordinates: {len(geo_valid)} / {len(df)}")

plt.figure(figsize=(14, 6))
scatter = plt.scatter(
    geo_valid['Longitude'], geo_valid['Latitude'], 
    c=geo_valid['Aggregate rating'], cmap='Spectral_r', 
    alpha=0.6, s=20
)
plt.colorbar(scatter, label='Aggregate Rating')
plt.title('Global Geospatial Distribution of Restaurants & Ratings', fontsize=14, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


In [ ]:
# 3.2 Correlation between Location and Rating
corr_lat = df['Latitude'].corr(df['Aggregate rating'])
corr_lon = df['Longitude'].corr(df['Aggregate rating'])
print(f"Latitude vs Aggregate Rating Correlation: {corr_lat:.4f}")
print(f"Longitude vs Aggregate Rating Correlation: {corr_lon:.4f}")


### Level 1 Summary of Findings:
- **Missing Values:** Only 9 records in `Cuisines` had missing values, which were safely imputed.
- **Target Distribution:** Significant zero-rating spike (22.49% unrated restaurants); ratings for active restaurants follow an approximately normal distribution centered around 3.2 - 3.4.
- **Geographic Concentration:** Over 90% of restaurants in the dataset are based in India (New Delhi, Gurgaon, Noida). North Indian and Chinese are the most ubiquitous cuisines.
